# 00. Запуск ETL

Запускает ETL на `HI-Small_Trans.csv`: первый день идёт только в историю счетов, разреженный хвост симуляции исключается. Каждый запуск создаёт отдельную папку и публикует `artifacts/runs/active.json` только после успешной записи.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").exists() and (p / "notebooks").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Запустите notebook из корня проекта или notebooks")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from aml.cli.etl import run_etl
from aml.data.read import load_dataset_splits
import pyarrow.parquet as pq
from time import perf_counter

In [ ]:
SOURCE = PROJECT_ROOT / "data" / "raw" / "HI-Small_Trans.csv"
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "runs"
SMOKE = False
if not SOURCE.is_file():
    raise FileNotFoundError(SOURCE)
print("Source:", SOURCE)
print("Size, MiB:", round(SOURCE.stat().st_size / 1024**2, 1))
print("Full dataset:", not SMOKE)

Source: /Users/artem/Documents/AML/AML/data/raw/HI-Small_Trans.csv
Size, MiB: 453.6
Full dataset: True


## Extract → validate → transform → load

Создаёт календарные периоды warmup → train → val → threshold → test. Исторические признаки используют только события с более ранним timestamp.

In [3]:
started = perf_counter()
RUN_DIR = run_etl(SOURCE, OUTPUT_DIR, smoke=SMOKE)
print("Completed run:", RUN_DIR)
print("Elapsed, minutes:", round((perf_counter() - started) / 60, 2))

Completed run: /Users/artem/Documents/AML/AML/artifacts/runs/20261014T145307Z_a2f6be14ec75
Elapsed, minutes: 0.7


## Проверка сохранённых файлов

Проверяем количество строк по Parquet metadata, не загружая все splits повторно в RAM.

In [4]:
manifest = json.loads((RUN_DIR / "data_manifest.json").read_text())
assert (RUN_DIR / "SUCCESS").is_file()
assert Path(manifest["source"]["path"]) == SOURCE.resolve()
configuration = json.loads((RUN_DIR / "configuration.json").read_text())
assert configuration["smoke"] is False
rows = []
for name in ("train", "val", "threshold", "test"):
    count = pq.ParquetFile(RUN_DIR / "data" / f"{name}.parquet").metadata.num_rows
    time_count = pq.ParquetFile(RUN_DIR / "data" / f"{name}_time.parquet").metadata.num_rows
    assert count == time_count > 0
    rows.append({"split": name, "rows": count, "timestamp_rows": time_count})
excluded = manifest["transformation"]["excluded_rows"]
assert sum(row["rows"] for row in rows) + sum(excluded.values()) == manifest["source"]["extracted_rows"]
display(pd.DataFrame(manifest["transformation"]["periods"]).T)
print("Excluded rows:", excluded)
display(pd.Series(json.loads((RUN_DIR / "quality_report.json").read_text())))
print("Next: 03_training.ipynb; optional: 01_eda.ipynb and 02_feature_engineering.ipynb")

,rows,min_timestamp,max_timestamp,positives
warmup,1114921,2022-09-01T00:00:00,2022-09-01T23:59:00,322
train,2134000,2022-09-02T00:00:00,2022-09-06T23:59:00,2208
val,482751,2022-09-07T00:00:00,2022-09-07T23:59:00,497
threshold,482773,2022-09-08T00:00:00,2022-09-08T23:59:00,539
test,862792,2022-09-09T00:00:00,2022-09-10T23:59:00,956
tail,1108,2022-09-11T00:05:00,2022-09-18T16:18:00,655


Excluded rows: {'warmup': 1114921, 'tail': 1108}


rows                                                         5078345
duplicate_rows                                                     9
zero_amounts                                                       0
self_transfers                                                591212
negatives                                                    5073168
positives                                                       5177
positive_rate                                               0.001019
min_timestamp                                    2022-09-01T00:00:00
max_timestamp                                    2022-09-18T16:18:00
payment_formats    {'ACH': 600797, 'Bitcoin': 146091, 'Cash': 490...
currencies         {'Australian Dollar': 275280, 'Bitcoin': 29421...
daily              [{'day': '2022-09-01', 'rows': 1114921, 'posit...
dtype: object

Next: 03_training.ipynb; optional: 01_eda.ipynb and 02_feature_engineering.ipynb
